# House Price Prediction
Kaggle: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

Workflow: load data → EDA → preprocessing → model comparison (CV R²) → final model → `submission.csv`.

> Put `train.csv` and `test.csv` in the `data/` folder before running.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from src.pipeline import (load_data, split_xy, get_models, make_pipeline,
                          MODEL_DIR, OUT_DIR, RANDOM_STATE)

sns.set_theme(style="whitegrid")
MODEL_DIR.mkdir(exist_ok=True); OUT_DIR.mkdir(exist_ok=True)

## 1. Load data

In [ ]:
train, test = load_data()
print("train:", train.shape, "| test:", test.shape)
train.head()

In [ ]:
train.info()

## 2. Exploratory data analysis

In [ ]:
train["SalePrice"].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train["SalePrice"], kde=True, ax=ax[0]).set_title("SalePrice")
sns.histplot(np.log1p(train["SalePrice"]), kde=True, ax=ax[1]).set_title("log1p(SalePrice)")
plt.show()
print("Skewness:", train["SalePrice"].skew().round(2), "-> after log:", np.log1p(train["SalePrice"]).skew().round(2))

In [ ]:
missing = (train.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
plt.figure(figsize=(10, 5))
sns.barplot(x=missing.values, y=missing.index, color="steelblue")
plt.xlabel("% missing"); plt.title("Missing values (train)")
plt.show()

In [ ]:
corr = train.select_dtypes("number").corr()
top = corr["SalePrice"].abs().sort_values(ascending=False).head(11).index
plt.figure(figsize=(9, 7))
sns.heatmap(train[top].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Top correlated features with SalePrice")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=train, x="OverallQual", y="SalePrice", ax=ax[0])
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice", ax=ax[1], alpha=0.6)
plt.show()

## 3. Preprocessing
* the 2 well-known outliers (`GrLivArea > 4000` but `SalePrice < 300000`, visible in the scatter plot above) are removed from training - they badly distort every model
* columns with more than 80 % missing values are dropped
* `MSSubClass` is a building-type code, so it is treated as a category
* numeric: median imputation + standardisation
* categorical: fill with `"Missing"`, one-hot encode, rare categories (<1 %) grouped together
* target is modelled as `log1p(SalePrice)` (predictions are converted back automatically)

All of this lives in one scikit-learn `Pipeline`, so the same steps are applied to train and test with no leakage.

In [ ]:
X, y, X_test, test_ids, dropped = split_xy(train, test)
print("Dropped columns:", dropped)
print("X:", X.shape, "| X_test:", X_test.shape)

## 4. Compare models (5-fold cross-validated R²)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = {}
for name, model in get_models().items():
    s = cross_val_score(make_pipeline(model, X), X, y, cv=cv, scoring="r2")
    scores[name] = s.mean()
    print(f"{name:<18} R2 = {s.mean():.4f} (+/- {s.std():.4f})")

results = pd.Series(scores).sort_values()
results.plot.barh(figsize=(8, 5), title="Cross-validated R²")
plt.xlim(max(0, results.min() - 0.05), 1)
plt.show()

## 5. Final model: GradientBoostingRegressor

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
check = make_pipeline(get_models()["GradientBoosting"], X_tr).fit(X_tr, y_tr)
pred = check.predict(X_val)
print(f"Hold-out R2  = {r2_score(y_val, pred):.4f}")
print(f"Hold-out RMSE = {np.sqrt(mean_squared_error(y_val, pred)):,.0f}")

plt.figure(figsize=(5, 5))
plt.scatter(y_val, pred, s=10, alpha=0.6)
lim = [y_val.min(), y_val.max()]
plt.plot(lim, lim, "r--"); plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.show()

In [ ]:
final = make_pipeline(get_models()["GradientBoosting"], X).fit(X, y)
joblib.dump(final, MODEL_DIR / "gbr.pkl")
print("Saved", MODEL_DIR / "gbr.pkl")

## 6. Predict test set and write `submission.csv`

In [ ]:
submission = pd.DataFrame({"Id": test_ids, "SalePrice": final.predict(X_test)})
submission.to_csv(OUT_DIR / "submission.csv", index=False)
submission.head()